In [1]:
!pip install pyg-library torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.5.0+${CUDA}.html

Looking in links: https://data.pyg.org/whl/torch-2.5.0+.html


In [2]:
!pip install torch_geometric

In [3]:
import os.path as osp
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from torch_geometric.nn import Node2Vec
from torch_geometric.utils import negative_sampling
from torch_geometric.datasets import Planetoid
import torch_geometric.transforms as T
from torch_geometric.nn import GCNConv
from torch_geometric.utils import train_test_split_edges

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [5]:
df = pd.read_csv("./DTO_cleaned.csv")
df = df.replace("","\n",regex=True)

In [6]:
def write_to_file(data, file_path, separator='`'):
    """
    Write data to a file iteratively.

    Parameters:
        data (list of lists): Data to be written (row-wise).
        file_path (str): Path to the output file.
        separator (str): Separator for columns.
    """
    with open(file_path, 'w') as file:  # Open in write mode
        for _, row in data.iterrows():  # Iterate over each row in the DataFrame
            line = separator.join(map(str, row))  # Convert row values to strings and join with separator
            file.write(line + '\n')  # Write the line and add a newline
write_to_file(df, './train1.txt')

In [7]:
import csv

def check_file(input_file, output_file):
    """
    Reads a .csv file, splits each line by `, and writes only those lines with exactly 3 elements to a new .csv file.

    Parameters:
        input_file (str): Path to the input .csv file.
        output_file (str): Path to the output .csv file.
    """
    with open(input_file, 'r') as infile, open(output_file, 'w', newline='') as outfile:
        reader = csv.reader(infile, delimiter='`')  # Read lines split by backtick
        writer = csv.writer(outfile, delimiter='`')  # Write lines split by backtick
        cnt =0
        tot =0
        for row in reader:
            if len(row) == 3:  # Check if the row has exactly 3 elements
                writer.writerow(row)  # Write valid rows to the output file
            else:
                cnt+=1
            tot+=1
                # print(row)
        print(cnt, f'for {input_file}', tot)
        

check_file('./train1.txt','./train.txt')

684 for ./train1.txt 123219


In [8]:
import torch
from torch_geometric.data import Data

# Step 1: Load and parse the .txt file
file_path = './train.txt'  # Replace with your file path
data = []
with open(file_path, 'r') as file:
    for line in file:
        try:
            subject, predicate, obj = line.strip().split('`')
            data.append((subject, predicate, obj))
        except: 
            print(line)

# Step 2: Map entities (subjects/objects) and predicates to indices
entities = set()
relations = set()

for subject, predicate, obj in data:
    entities.add(subject)
    entities.add(obj)
    relations.add(predicate)

entities = list(entities)
relations = list(relations)

entities.sort()
relations.sort()

entity_to_idx = {entity: idx for idx, entity in enumerate(entities)}
relation_to_idx = {relation: idx for idx, relation in enumerate(relations)}

# Step 3: Create edge index and edge attributes
edge_index = []
edge_attr = []

for subject, predicate, obj in data:
    src = entity_to_idx[subject]
    dst = entity_to_idx[obj]
    rel = relation_to_idx[predicate]
    
    edge_index.append((src, dst))
    edge_attr.append(rel)

# Convert to PyTorch tensors
edge_index = torch.tensor(edge_index).t().contiguous()  # Shape [2, num_edges]
edge_attr = torch.tensor(edge_attr)  # Shape [num_edges]

# Step 4: Create PyTorch Geometric Data object
num_nodes = len(entities)
graph_data = Data(edge_index=edge_index, edge_attr=edge_attr, num_nodes=num_nodes)

print(graph_data)


Data(edge_index=[2, 122535], edge_attr=[122535], num_nodes=83717)


In [9]:
model = Node2Vec(graph_data.edge_index,embedding_dim=120,walk_length=20,context_size=10,walks_per_node=20,num_negative_samples=1,p=1,q=1,sparse=True).to(device)

In [10]:
loader = model.loader(batch_size=128, shuffle=True, num_workers=4)
optimizer = torch.optim.SparseAdam(list(model.parameters()), lr=0.01)

In [11]:
!pip install tqdm

In [12]:
from tqdm.notebook import tqdm

In [13]:
def train():
    model.train()
    total_loss = 0
    for pos_rw, neg_rw in tqdm(loader):
        optimizer.zero_grad()
        loss = model.loss(pos_rw.to(device), neg_rw.to(device))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [14]:
@torch.no_grad()
def test():
    model.eval()
    z = model()
    acc = model.test(z[data.train_mask], data.y[data.train_mask],
                     z[data.test_mask], data.y[data.test_mask],
                     max_iter=150)
    return acc

In [15]:
data = graph_data

In [16]:
# train_ratio = 0.8
# num_train_nodes = int(train_ratio * num_nodes)
# perm = torch.randperm(num_nodes)  # Shuffle node indices

# train_mask = torch.zeros(num_nodes, dtype=torch.bool)
# test_mask = torch.zeros(num_nodes, dtype=torch.bool)

# train_mask[perm[:num_train_nodes]] = True
# test_mask[perm[num_train_nodes:]] = True

# # Add masks to the Data object
# data.train_mask = train_mask
# data.test_mask = test_mask

# print(data)

In [17]:
# data.train_mask = data.val_mask = data.test_mask = data.y = None
# data = train_test_split(data)

In [18]:
data

Data(edge_index=[2, 122535], edge_attr=[122535], num_nodes=83717)

In [19]:
data
data.y
# data.y.unique()
for epoch in range(1,11):
    loss = train()
    # acc = test()
    print(f'Epoch: {epoch:02d}, Loss: {loss:.4f}')

  0%|          | 0/655 [00:00<?, ?it/s]

Epoch: 01, Loss: 2.7945


  0%|          | 0/655 [00:00<?, ?it/s]

Epoch: 02, Loss: 0.9232


  0%|          | 0/655 [00:00<?, ?it/s]

...

Epoch: 08, Loss: 0.7464


  0%|          | 0/655 [00:00<?, ?it/s]

Epoch: 09, Loss: 0.7469


  0%|          | 0/655 [00:00<?, ?it/s]

Epoch: 10, Loss: 0.7478


In [21]:
torch.save(model.state_dict(), 'model.pt')

In [32]:
model(torch.tensor([1]))

tensor([[ 0.2587,  0.2928,  0.1150,  0.0418,  0.1717, -0.0160, -0.2005,  0.4667,
          0.2310, -0.0734, -0.2427, -0.0643, -0.1533, -0.2167, -0.0739,  0.0845,
         -0.2606,  0.2685, -0.0801, -0.1068,  0.1287, -0.0800,  0.0087, -0.4213,
          0.2359,  0.0512, -0.2684,  0.0784, -0.0148,  0.1579, -0.3189, -0.2309,
         -0.0589,  0.0379, -0.0941, -0.1042,  0.2515,  0.2706, -0.3448,  0.1961,
          0.0221,  0.1027, -0.0889,  0.1699,  0.0671, -0.0345, -0.2329,  0.1157,
         -0.0655, -0.3920, -0.0556, -0.0928,  0.1524, -0.1378, -0.0006,  0.0686,
          0.3358,  0.0297,  0.0212, -0.0868,  0.1052,  0.1156, -0.1045, -0.0244,
          0.2314, -0.3427,  0.0469,  0.2077,  0.2518, -0.2693, -0.2249, -0.0432,
         -0.0368, -0.0813,  0.1803, -0.1575,  0.0332, -0.0950, -0.0939,  0.3396,
         -0.4039, -0.1394,  0.2167,  0.0529,  0.1532, -0.0328,  0.2562, -0.1002,
          0.0883, -0.1278, -0.0303, -0.1269, -0.1063, -0.0171, -0.0647, -0.0188,
         -0.3577,  0.3373, -

In [37]:
len(entities)

83717

In [36]:
model().shape

torch.Size([83717, 120])